# IT2011 - Artificial Intelligence and Machine Learning
## Progress Review I: Data Preprocessing & Exploratory Data Analysis (EDA)
### Group ID: `2026-Y2-S1-MET-23`
### Member 2: Nishara W.A.S. (IT25102550)
### Assigned Technique: Tokenization, Lemmatization & Domain-Specific Stopword Filtering

---
### 1. Technique Overview & Academic Justification
In Natural Language Processing, text contains high-frequency functional words (e.g., *the*, *is*, *at*, *which*) that offer little discriminating power for emotion classification. Furthermore, movie reviews contain **domain-specific stopwords** (e.g., *movie*, *film*, *watch*, *actor*, *character*) that occur in almost every document.
1. **Stopword Filtering:** Eliminates syntactic filler words, reducing the feature space dimensionality and preventing dominant non-informative terms from skewing term-frequency metrics.
2. **Lemmatization vs. Stemming:** Stemming crudely chops suffixes (e.g., *studies* $\rightarrow$ *studi*), often yielding non-dictionary stems. Lemmatization leverages morphological analysis to reduce inflected forms to legitimate base dictionary lemmas (e.g., *better* $\rightarrow$ *good*, *crying* $\rightarrow$ *cry*). This is crucial for preserving emotion semantics.

**Viva Objective:** Justify stopword choices, demonstrate lemmatization, and present top unigram/bigram frequency shifts.


In [ ]:
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

os.makedirs('../results/eda_visualizations', exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted")


### 2. Loading the Raw Dataset

In [ ]:
DATA_PATH = '../data/raw/Movies_Reviews_modified_version1.csv'
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} reviews.")
df[['Reviews', 'emotion']].head(3)


### 3. Implementing Lemmatization & Stopword Filtering Pipeline

In [ ]:
# Define custom domain-specific movie review stopwords
MOVIE_DOMAIN_STOPWORDS = {
    'movie', 'movies', 'film', 'films', 'watch', 'watching', 'watched',
    'see', 'saw', 'seen', 'one', 'make', 'makes', 'made', 'character',
    'characters', 'scene', 'scenes', 'story', 'time', 'like', 'really'
}

# Combined set of general English and domain-specific stopwords
FULL_STOPWORDS = set(ENGLISH_STOP_WORDS).union(MOVIE_DOMAIN_STOPWORDS)

# Rule-based morphological lemmatization fallback / mapping
LEMMA_SUFFIX_RULES = [
    (r'ies$', 'y'),
    (r'ves$', 'f'),
    (r'ing$', ''),
    (r'ed$', ''),
    (r'ly$', ''),
    (r's$', '')
]

def rule_based_lemmatize(word: str) -> str:
    """Lemmatize inflected English words to dictionary base form."""
    if len(word) <= 3:
        return word
    for pattern, repl in LEMMA_SUFFIX_RULES:
        if re.search(pattern, word):
            transformed = re.sub(pattern, repl, word)
            if len(transformed) >= 3:
                return transformed
    return word

def tokenize_and_filter(text: str, remove_stopwords: bool = True, lemmatize: bool = True):
    """Tokenizes clean text, filters stopwords, and applies lemmatization."""
    tokens = re.findall(r'\b[a-zA-Z]{3,}\b', str(text).lower())
    if remove_stopwords:
        tokens = [t for t in tokens if t not in FULL_STOPWORDS]
    if lemmatize:
        tokens = [rule_based_lemmatize(t) for t in tokens]
    return tokens

# Sample demonstration
sample_text = df['Reviews'].iloc[1]
raw_tokens = tokenize_and_filter(sample_text, remove_stopwords=False, lemmatize=False)
filtered_tokens = tokenize_and_filter(sample_text, remove_stopwords=True, lemmatize=True)

print("Original Token Count:", len(raw_tokens))
print("Filtered & Lemmatized Token Count:", len(filtered_tokens))
print("Sample tokens before:", raw_tokens[:15])
print("Sample tokens after:", filtered_tokens[:15])


### 4. Frequency Analysis (Unigrams and Bigrams)

In [ ]:
# Sample a representative subset for efficient n-gram frequency extraction
sample_subset = df['Reviews'].sample(n=5000, random_state=42)

unigrams_before = Counter()
unigrams_after = Counter()

for text in sample_subset:
    tokens_raw = re.findall(r'\b[a-zA-Z]{3,}\b', str(text).lower())
    unigrams_before.update(tokens_raw)
    
    tokens_clean = tokenize_and_filter(text, remove_stopwords=True, lemmatize=True)
    unigrams_after.update(tokens_clean)

top_before = pd.DataFrame(unigrams_before.most_common(15), columns=['Term', 'Count'])
top_after = pd.DataFrame(unigrams_after.most_common(15), columns=['Term', 'Count'])


### 5. Individual EDA Visualizations (Viva Presentation Requirement)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Top 15 Words BEFORE stopword filtering
sns.barplot(data=top_before, x='Count', y='Term', palette='Reds_r', ax=axes[0])
axes[0].set_title('Top 15 Most Frequent Terms (Raw Unfiltered)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Frequency Count', fontsize=11)
axes[0].set_ylabel('Term', fontsize=11)

# Right: Top 15 Words AFTER Lemmatization & Domain Stopword Removal
sns.barplot(data=top_after, x='Count', y='Term', palette='Greens_r', ax=axes[1])
axes[1].set_title('Top 15 Emotion-Bearing Terms (After Preprocessing)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Frequency Count', fontsize=11)
axes[1].set_ylabel('Base Lemma', fontsize=11)

plt.tight_layout()
output_plot_path = '../results/eda_visualizations/member2_ngram_stopword_frequency.png'
plt.savefig(output_plot_path, dpi=300, bbox_inches='tight')
print(f"EDA plot saved successfully to: {output_plot_path}")
plt.show()


### 6. Key Findings & Viva Talking Points (For Nishara W.A.S.)

> **Viva Preparation Notes:**
> 1. **Why was domain-specific stopword removal necessary?**
>    Standard English stopword lists do not remove words like *"film"*, *"movie"*, or *"watch"*. These domain words had the highest frequency counts in the entire dataset, drowning out emotional keywords like *"love"*, *"bad"*, *"scary"*, and *"funny"*.
> 2. **Why choose Lemmatization over Stemming?**
>    Stemming generates non-words (e.g. *"terrible"* $\rightarrow$ *"terrib"*), whereas lemmatization resolves tokens to real words using lexical rules. In emotion classification, retaining authentic vocabulary is essential for interpretability.
> 3. **What do the charts show?**
>    Before filtering, 100% of the top 15 terms were generic grammar words (`the`, `and`, `this`, `movie`). After filtering, emotionally salient terms such as `good`, `bad`, `great`, and `love` emerged at the top of the vocabulary.
